# Task 1 — Reasoning Trajectories (artifact v3)

Этот блокнот собирает YAML для экспертной траектории рассуждений и сразу поддерживает:

- читаемый интерфейс для **светлой и тёмной темы Google Colab**;
- **направленность ребра** и отдельную текстовую метку `direction_label`;
- флаг **одновременного открытия**;
- географию открытия: **сначала страна, потом город**, оба поля с **Wikidata QID**;
- классификацию по **отраслям науки и техники** с **Wikidata QID**;
- экспорт в формат **artifact_version: 3**.

Автоконвейер в репозитории затем использует эти поля вместе с автоподбором географии/отраслей из метаданных статей и Wikidata. Эксперт при этом может вручную исправить автоматический выбор.

In [ ]:
# @title Установка и импорт
import json, os, sys, re
from pathlib import Path


def run(cmd):
    import subprocess
    print('>', ' '.join(map(str, cmd)))
    subprocess.check_call(cmd)


run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'ipywidgets', 'pyyaml', 'requests', 'unidecode'])

import yaml
import requests
import ipywidgets as W
from IPython.display import display, HTML, Markdown, clear_output

try:
    from unidecode import unidecode
except Exception:
    unidecode = None

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    colab_files = None
    IN_COLAB = False


def find_repo_root() -> Path | None:
    candidates = []
    env_dir = os.environ.get('TPG_REPO_DIR')
    if env_dir:
        candidates.append(Path(env_dir))
    cwd = Path.cwd()
    candidates.extend([cwd, cwd.parent, Path('/content'), Path('/mnt/data')])
    for base in [Path('/content'), Path('/mnt/data'), cwd, cwd.parent]:
        if not base.exists():
            continue
        for pattern in ('top-papers-graph*', 'repo_mod', 'repo_fixed'):
            candidates.extend(base.glob(pattern))
    seen = set()
    for c in candidates:
        try:
            c = c.resolve()
        except Exception:
            pass
        key = str(c)
        if key in seen:
            continue
        seen.add(key)
        if (c / 'pyproject.toml').exists() and (c / 'src' / 'scireason').exists():
            return c
    return None


REPO_DIR = find_repo_root()
if REPO_DIR is not None:
    SRC_DIR = REPO_DIR / 'src'
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))

print('Готово. Запускайте следующую ячейку.')

In [ ]:
# @title Вспомогательные функции и CSS
WIKIDATA_API = 'https://www.wikidata.org/w/api.php'
WIKIDATA_USER_AGENT = 'TrajectoryFormColab/3.0 (educational notebook)'
QID_RE = re.compile(r'^Q\d+$', re.I)


def apply_css():
    display(HTML("""
    <style>
    :root {
      --t1-bg: #ffffff;
      --t1-card: #f8fafc;
      --t1-card-border: #dbe4ee;
      --t1-text: #0f172a;
      --t1-muted: #475569;
      --t1-accent: #2563eb;
    }
    @media (prefers-color-scheme: dark) {
      :root {
        --t1-bg: #111827;
        --t1-card: #1f2937;
        --t1-card-border: #374151;
        --t1-text: #f8fafc;
        --t1-muted: #cbd5e1;
        --t1-accent: #60a5fa;
      }
    }
    .t1-card, .t1-note, .t1-chip {
      color: var(--t1-text);
      font-family: Inter, system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    }
    .t1-card {
      background: var(--t1-card);
      border: 1px solid var(--t1-card-border);
      border-radius: 16px;
      padding: 14px 16px;
      margin: 8px 0;
      box-shadow: 0 1px 2px rgba(15, 23, 42, 0.06);
    }
    .t1-card h3, .t1-card h4 { margin: 0 0 8px 0; }
    .t1-note {
      background: color-mix(in srgb, var(--t1-card) 88%, var(--t1-accent) 12%);
      border: 1px solid var(--t1-card-border);
      border-radius: 12px;
      padding: 10px 12px;
      margin: 8px 0;
      color: var(--t1-muted);
    }
    .t1-chip {
      display: inline-flex;
      gap: 8px;
      align-items: center;
      border-radius: 999px;
      border: 1px solid var(--t1-card-border);
      padding: 4px 10px;
      margin: 4px 6px 0 0;
      background: var(--t1-bg);
    }
    .t1-small { font-size: 12px; color: var(--t1-muted); }
    </style>
    """))


def _clean(s):
    return str(s or '').strip()


def slugify_name(s: str) -> str:
    raw = _clean(s)
    if not raw:
        return ''
    if unidecode is not None:
        raw = unidecode(raw)
    raw = raw.lower()
    raw = re.sub(r'[^a-z0-9]+', '_', raw)
    raw = re.sub(r'_+', '_', raw).strip('_')
    return raw


def wikidata_search(query: str, language: str = 'ru', limit: int = 10):
    query = _clean(query)
    if not query:
        return []
    params = {
        'action': 'wbsearchentities',
        'search': query,
        'language': language or 'ru',
        'format': 'json',
        'limit': int(limit or 10),
    }
    headers = {'User-Agent': WIKIDATA_USER_AGENT, 'Accept': 'application/json'}
    resp = requests.get(WIKIDATA_API, params=params, headers=headers, timeout=30)
    resp.raise_for_status()
    data = resp.json() or {}
    out = []
    for item in data.get('search', []) or []:
        out.append({
            'id': item.get('id'),
            'label': item.get('label') or item.get('display', {}).get('label', {}).get('value'),
            'description': item.get('description') or item.get('display', {}).get('description', {}).get('value') or '',
        })
    return out


def wikidata_option_label(item: dict) -> str:
    label = _clean(item.get('label')) or _clean(item.get('id'))
    desc = _clean(item.get('description'))
    qid = _clean(item.get('id'))
    return f'{qid} — {label}' + (f' — {desc}' if desc else '')


def make_single_wd_picker(title: str, placeholder: str, *, combine_with=None):
    state = {'value': None}
    query = W.Text(placeholder=placeholder, description=title, layout=W.Layout(width='60%'))
    btn = W.Button(description='Найти', icon='search')
    dropdown = W.Dropdown(options=[('—', '')], description='Вариант', layout=W.Layout(width='100%'))
    select_btn = W.Button(description='Выбрать', button_style='info')
    clear_btn = W.Button(description='Очистить')
    selected = W.HTML()

    def refresh_selected():
        value = state['value']
        if not value:
            selected.value = '<div class="t1-small">Ничего не выбрано</div>'
            return
        selected.value = f'<span class="t1-chip"><b>{value["id"]}</b><span>{value["label"]}</span></span>'

    def search_clicked(_):
        q = _clean(query.value)
        if combine_with is not None:
            extra = combine_with()
            if extra:
                q = f'{q}, {extra}' if q else extra
        try:
            rows = wikidata_search(q)
            dropdown.options = [(wikidata_option_label(r), json.dumps(r, ensure_ascii=False)) for r in rows] if rows else [('Ничего не найдено', '')]
        except Exception as e:
            dropdown.options = [(f'Ошибка поиска: {type(e).__name__}: {e}', '')]

    def choose_clicked(_):
        raw = dropdown.value
        state['value'] = json.loads(raw) if raw else None
        refresh_selected()

    def clear_clicked(_):
        state['value'] = None
        refresh_selected()

    btn.on_click(search_clicked)
    select_btn.on_click(choose_clicked)
    clear_btn.on_click(clear_clicked)
    refresh_selected()

    box = W.VBox([W.HBox([query, btn, select_btn, clear_btn]), dropdown, selected])
    box.get_value = lambda: state['value']
    box.set_value = lambda value: state.update({'value': value}) or refresh_selected()
    return box


def make_multi_wd_picker(title: str, placeholder: str):
    state = {'items': []}
    query = W.Text(placeholder=placeholder, description=title, layout=W.Layout(width='60%'))
    btn = W.Button(description='Найти', icon='search')
    dropdown = W.Dropdown(options=[('—', '')], description='Вариант', layout=W.Layout(width='100%'))
    add_btn = W.Button(description='Добавить', button_style='info')
    clear_btn = W.Button(description='Очистить всё')
    selected = W.HTML()

    def refresh_selected():
        if not state['items']:
            selected.value = '<div class="t1-small">Пока нет выбранных отраслей</div>'
            return
        selected.value = ''.join(
            f'<span class="t1-chip"><b>{item["id"]}</b><span>{item["label"]}</span></span>'
            for item in state['items']
        )

    def search_clicked(_):
        q = _clean(query.value)
        try:
            rows = wikidata_search(q)
            dropdown.options = [(wikidata_option_label(r), json.dumps(r, ensure_ascii=False)) for r in rows] if rows else [('Ничего не найдено', '')]
        except Exception as e:
            dropdown.options = [(f'Ошибка поиска: {type(e).__name__}: {e}', '')]

    def add_clicked(_):
        raw = dropdown.value
        if not raw:
            return
        item = json.loads(raw)
        seen = {x['id'] for x in state['items'] if isinstance(x, dict)}
        if item.get('id') not in seen:
            state['items'].append(item)
        refresh_selected()

    def clear_clicked(_):
        state['items'] = []
        refresh_selected()

    btn.on_click(search_clicked)
    add_btn.on_click(add_clicked)
    clear_btn.on_click(clear_clicked)
    refresh_selected()

    box = W.VBox([W.HBox([query, btn, add_btn, clear_btn]), dropdown, selected])
    box.get_value = lambda: list(state['items'])
    box.set_value = lambda values: state.update({'items': list(values or [])}) or refresh_selected()
    return box


def prune_empty(value):
    if isinstance(value, dict):
        out = {k: prune_empty(v) for k, v in value.items()}
        return {k: v for k, v in out.items() if v not in (None, '', [], {})}
    if isinstance(value, list):
        out = [prune_empty(v) for v in value]
        return [v for v in out if v not in (None, '', [], {})]
    return value


def validate_qid_dict(value, label):
    errs = []
    if not value:
        return errs
    if not isinstance(value, dict):
        return [f'{label}: ожидался объект Wikidata']
    qid = _clean(value.get('id'))
    if not qid or not QID_RE.fullmatch(qid):
        errs.append(f'{label}: отсутствует корректный QID')
    if not _clean(value.get('label')):
        errs.append(f'{label}: отсутствует метка')
    return errs


def make_expert_block():
    last = W.Text(description='Фамилия', layout=W.Layout(width='33%'))
    first = W.Text(description='Имя', layout=W.Layout(width='33%'))
    patronymic = W.Text(description='Отчество', layout=W.Layout(width='33%'))
    box = W.VBox([
        W.HTML('<div class="t1-card"><h3>Эксперт</h3><div class="t1-small">Поля нужны для корректного имени файла и карточки эксперта.</div></div>'),
        W.HBox([last, first, patronymic]),
    ])
    box.widgets = {'last': last, 'first': first, 'patronymic': patronymic}
    return box


def make_paper_row(index: int):
    pid = W.Text(description=f'Paper {index} id', layout=W.Layout(width='45%'))
    title = W.Text(description='Title', layout=W.Layout(width='45%'))
    year = W.IntText(description='Year', layout=W.Layout(width='18%'))
    return {'container': W.VBox([W.HBox([pid, year]), title]), 'widgets': {'id': pid, 'title': title, 'year': year}}


def make_source_row(step_idx: int, src_idx: int):
    stype = W.Dropdown(options=['text', 'image', 'table'], value='text', description=f'Source {src_idx}', layout=W.Layout(width='20%'))
    source = W.Text(description='ID/URL', layout=W.Layout(width='78%'))
    locator = W.Text(description='Locator', layout=W.Layout(width='32%'))
    snippet = W.Textarea(description='Snippet', layout=W.Layout(width='100%', height='90px'))
    row = W.VBox([W.HBox([stype, source]), locator, snippet])
    return {'container': row, 'widgets': {'type': stype, 'source': source, 'locator': locator, 'snippet': snippet}}


def make_step_row(step_idx: int):
    title = W.HTML(f'<div class="t1-card"><h3>Шаг {step_idx}</h3><div class="t1-small">Заполните источник, вывод и контекст открытия. Автоконвейер потом сможет использовать эти поля вместе с OpenAlex/Wikidata.</div></div>')
    claim = W.Textarea(description='Claim', layout=W.Layout(width='100%', height='90px'))
    cond_system = W.Text(description='System', layout=W.Layout(width='49%'))
    cond_env = W.Text(description='Environment', layout=W.Layout(width='49%'))
    cond_protocol = W.Textarea(description='Protocol', layout=W.Layout(width='100%', height='70px'))
    cond_notes = W.Textarea(description='Notes', layout=W.Layout(width='100%', height='70px'))

    sources_box = W.VBox([])
    source_rows = []
    add_source_btn = W.Button(description='Добавить источник', icon='plus')
    remove_source_btn = W.Button(description='Удалить последний источник', icon='minus')

    def refresh_sources():
        sources_box.children = [r['container'] for r in source_rows]

    def add_source(_=None):
        source_rows.append(make_source_row(step_idx, len(source_rows) + 1))
        refresh_sources()

    def remove_source(_=None):
        if source_rows:
            source_rows.pop()
            refresh_sources()

    add_source_btn.on_click(add_source)
    remove_source_btn.on_click(remove_source)
    add_source()

    simultaneous = W.Checkbox(value=False, description='Одновременное открытие / независимое параллельное открытие')
    country_picker = make_single_wd_picker('Страна', 'например, Germany / Германия')
    city_picker = make_single_wd_picker('Город', 'например, Munich / Мюнхен', combine_with=lambda: (country_picker.get_value() or {}).get('label', ''))
    branches_picker = make_multi_wd_picker('Отрасль', 'например, physics / materials science')

    inference = W.Textarea(description='Inference', layout=W.Layout(width='100%', height='90px'))
    next_question = W.Textarea(description='Next question', layout=W.Layout(width='100%', height='90px'))

    container = W.VBox([
        title,
        claim,
        W.HBox([cond_system, cond_env]),
        cond_protocol,
        cond_notes,
        W.HTML('<div class="t1-note">Источники: можно добавлять текст, изображения и таблицы. Для figure/table используйте locator, например Figure 2 или Table S1.</div>'),
        W.HBox([add_source_btn, remove_source_btn]),
        sources_box,
        W.HTML('<div class="t1-note">Контекст открытия: сначала выберите страну, потом город. Автоконвейер позже сможет предложить такие поля автоматически из аффилиаций и Wikidata, но эксперт всегда может исправить вручную.</div>'),
        simultaneous,
        country_picker,
        city_picker,
        branches_picker,
        inference,
        next_question,
    ])

    return {
        'container': container,
        'widgets': {
            'claim': claim,
            'cond_system': cond_system,
            'cond_env': cond_env,
            'cond_protocol': cond_protocol,
            'cond_notes': cond_notes,
            'simultaneous': simultaneous,
            'country': country_picker,
            'city': city_picker,
            'branches': branches_picker,
            'inference': inference,
            'next_question': next_question,
        },
        'source_rows': source_rows,
    }


def make_edge_row(index: int, step_options_fn):
    step_options = step_options_fn()
    frm = W.Dropdown(options=step_options, description=f'Edge {index}: from', layout=W.Layout(width='24%'))
    to = W.Dropdown(options=step_options, description='to', layout=W.Layout(width='24%'))
    predicate = W.Text(value='leads_to', description='Predicate', layout=W.Layout(width='24%'))
    directionality = W.Dropdown(
        options=[('Directed', 'directed'), ('Bidirectional', 'bidirectional'), ('Simultaneous', 'simultaneous')],
        value='directed',
        description='Direction',
        layout=W.Layout(width='28%'),
    )
    direction_label = W.Text(description='Direction label', placeholder='например, from step 2 to step 5', layout=W.Layout(width='68%'))
    simultaneous = W.Checkbox(value=False, description='Одновременное открытие')
    container = W.VBox([
        W.HBox([frm, to, predicate, directionality]),
        W.HBox([direction_label, simultaneous]),
    ])
    return {'container': container, 'widgets': {'from': frm, 'to': to, 'predicate': predicate, 'directionality': directionality, 'direction_label': direction_label, 'simultaneous': simultaneous}}


apply_css()
print('Готово. Запускайте следующую ячейку.')

In [ ]:
# @title Форма Task 1 (artifact v3)
expert_block = make_expert_block()

local_domain_configs = []
if REPO_DIR is not None:
    domain_dir = REPO_DIR / 'configs' / 'domains'
    for p in sorted(domain_dir.glob('*.yaml')):
        try:
            raw = yaml.safe_load(p.read_text(encoding='utf-8')) or {}
            local_domain_configs.append({
                'id': raw.get('wikidata_qid') or raw.get('domain_id') or p.stem,
                'label': raw.get('title') or p.stem,
                'description': raw.get('domain_id') or p.stem,
            })
        except Exception:
            continue

domain_query = W.Text(description='Domain', placeholder='например, Science / Materials science', layout=W.Layout(width='55%'))
domain_search_btn = W.Button(description='Найти домен', icon='search')
domain_results = W.Dropdown(options=[('—', '')], description='Вариант', layout=W.Layout(width='100%'))
domain_select_btn = W.Button(description='Выбрать', button_style='info')
domain_selected = W.HTML()
_selected_domain = {'value': None}


def refresh_domain_selected():
    value = _selected_domain['value']
    if not value:
        domain_selected.value = '<div class="t1-small">Домен не выбран</div>'
        return
    domain_selected.value = f'<span class="t1-chip"><b>{value["id"]}</b><span>{value["label"]}</span></span>'


def search_domain(_):
    q = _clean(domain_query.value)
    rows = []
    if q:
        qlow = q.lower()
        rows.extend([x for x in local_domain_configs if qlow in _clean(x['label']).lower() or qlow in _clean(x['description']).lower()])
    try:
        rows.extend(wikidata_search(q, limit=8))
    except Exception as e:
        rows.append({'id': '', 'label': f'Ошибка поиска: {type(e).__name__}: {e}', 'description': ''})
    dedup = []
    seen = set()
    for r in rows:
        key = (r.get('id'), r.get('label'))
        if key in seen:
            continue
        seen.add(key)
        dedup.append(r)
    domain_results.options = [(wikidata_option_label(r), json.dumps(r, ensure_ascii=False)) for r in dedup] if dedup else [('Ничего не найдено', '')]


def choose_domain(_):
    raw = domain_results.value
    _selected_domain['value'] = json.loads(raw) if raw else None
    refresh_domain_selected()


domain_search_btn.on_click(search_domain)
domain_select_btn.on_click(choose_domain)
refresh_domain_selected()

topic = W.Textarea(description='Topic', layout=W.Layout(width='100%', height='110px'))
cutoff_year = W.IntText(description='Cutoff year', value=2025)
output_name = W.Text(description='File name', value='expert_trajectory_v3.yaml', layout=W.Layout(width='100%'))
status = W.HTML()
export_out = W.Output()

paper_rows = []
papers_box = W.VBox([])
add_paper_btn = W.Button(description='Добавить публикацию', icon='plus')
remove_paper_btn = W.Button(description='Удалить последнюю публикацию', icon='minus')


def refresh_papers():
    papers_box.children = [r['container'] for r in paper_rows]


def add_paper(_=None):
    paper_rows.append(make_paper_row(len(paper_rows) + 1))
    refresh_papers()


def remove_paper(_=None):
    if paper_rows:
        paper_rows.pop()
        refresh_papers()


add_paper_btn.on_click(add_paper)
remove_paper_btn.on_click(remove_paper)
add_paper()

step_rows = []
steps_box = W.VBox([])
add_step_btn = W.Button(description='Добавить шаг', icon='plus', button_style='success')
remove_step_btn = W.Button(description='Удалить последний шаг', icon='minus')


def step_options():
    return [(f'Шаг {i+1}', i + 1) for i in range(len(step_rows))] or [('—', 1)]


edge_rows = []
edges_box = W.VBox([])
add_edge_btn = W.Button(description='Добавить ребро', icon='plus')
remove_edge_btn = W.Button(description='Удалить последнее ребро', icon='minus')


def refresh_edges_options():
    opts = step_options()
    for row in edge_rows:
        row['widgets']['from'].options = opts
        row['widgets']['to'].options = opts


def refresh_steps():
    steps_box.children = [r['container'] for r in step_rows]
    refresh_edges_options()


def add_step(_=None):
    step_rows.append(make_step_row(len(step_rows) + 1))
    refresh_steps()


def remove_step(_=None):
    if step_rows:
        step_rows.pop()
        refresh_steps()


add_step_btn.on_click(add_step)
remove_step_btn.on_click(remove_step)
add_step()


def refresh_edges():
    edges_box.children = [r['container'] for r in edge_rows]
    refresh_edges_options()


def add_edge(_=None):
    edge_rows.append(make_edge_row(len(edge_rows) + 1, step_options))
    refresh_edges()


def remove_edge(_=None):
    if edge_rows:
        edge_rows.pop()
        refresh_edges()


add_edge_btn.on_click(add_edge)
remove_edge_btn.on_click(remove_edge)


def build_expert_payload():
    w = expert_block.widgets
    last = _clean(w['last'].value)
    first = _clean(w['first'].value)
    patronymic = _clean(w['patronymic'].value)
    full_name = ' '.join(x for x in [last, first, patronymic] if x)
    latin_parts = [slugify_name(x).replace('_', ' ').title() for x in [first, patronymic, last] if _clean(x)]
    latin_full = ' '.join(x for x in latin_parts if x)
    latin_slug = slugify_name('_'.join(x for x in [last, first, patronymic] if _clean(x)))
    return prune_empty({
        'last_name': last,
        'first_name': first,
        'patronymic': patronymic,
        'full_name': full_name,
        'latin_full_name': latin_full,
        'latin_slug': latin_slug,
    })


def build_steps_payload():
    out = []
    for i, step in enumerate(step_rows, start=1):
        w = step['widgets']
        sources = []
        for s in step['source_rows']:
            sw = s['widgets']
            sources.append(prune_empty({
                'type': _clean(sw['type'].value),
                'source': _clean(sw['source'].value),
                'locator': _clean(sw['locator'].value),
                'snippet_or_summary': _clean(sw['snippet'].value),
            }))
        discovery_context = prune_empty({
            'simultaneous_discovery': bool(w['simultaneous'].value),
            'geography': {
                'country': w['country'].get_value(),
                'city': w['city'].get_value(),
            },
            'science_branches': w['branches'].get_value(),
        })
        out.append(prune_empty({
            'step_id': i,
            'claim': _clean(w['claim'].value),
            'conditions': {
                'system': _clean(w['cond_system'].value),
                'environment': _clean(w['cond_env'].value),
                'protocol': _clean(w['cond_protocol'].value),
                'notes': _clean(w['cond_notes'].value),
            },
            'sources': sources,
            'discovery_context': discovery_context,
            'inference': _clean(w['inference'].value),
            'next_question': _clean(w['next_question'].value),
        }))
    return out


def build_edges_payload():
    out = []
    for row in edge_rows:
        w = row['widgets']
        out.append(prune_empty({
            'from_step_id': int(w['from'].value),
            'to_step_id': int(w['to'].value),
            'predicate': _clean(w['predicate'].value) or 'leads_to',
            'directionality': _clean(w['directionality'].value) or 'directed',
            'direction_label': _clean(w['direction_label'].value),
            'simultaneous_discovery': bool(w['simultaneous'].value),
        }))
    return out


def build_document():
    expert = build_expert_payload()
    submission_id = expert.get('latin_slug') or slugify_name(_clean(topic.value)) or 'trajectory_submission'
    doc = prune_empty({
        'artifact_version': 3,
        'domain': (_selected_domain['value'] or {}).get('id'),
        'topic': _clean(topic.value),
        'cutoff_year': int(cutoff_year.value) if cutoff_year.value else None,
        'submission_id': submission_id,
        'papers': [prune_empty({'id': _clean(r['widgets']['id'].value), 'title': _clean(r['widgets']['title'].value), 'year': int(r['widgets']['year'].value) if r['widgets']['year'].value else None}) for r in paper_rows],
        'steps': build_steps_payload(),
        'edges': build_edges_payload(),
        'expert': expert,
    })
    return doc


def validate_doc(doc):
    errs = []
    if not _clean(doc.get('topic')):
        errs.append('Не заполнено поле topic')
    domain_qid = _clean(doc.get('domain'))
    if not QID_RE.fullmatch(domain_qid or ''):
        errs.append('Домен должен быть выбран через Wikidata и сохранён как QID')
    steps = doc.get('steps') or []
    if not steps:
        errs.append('Нужен хотя бы один шаг')
    for i, step in enumerate(steps, start=1):
        if not _clean(step.get('claim')):
            errs.append(f'Шаг {i}: отсутствует claim')
        if not _clean(step.get('inference')):
            errs.append(f'Шаг {i}: отсутствует inference')
        if not _clean(step.get('next_question')):
            errs.append(f'Шаг {i}: отсутствует next_question')
        sources = step.get('sources') or []
        if not sources:
            errs.append(f'Шаг {i}: нужен хотя бы один source')
        for j, src in enumerate(sources, start=1):
            if _clean(src.get('type')) not in {'text', 'image', 'table'}:
                errs.append(f'Шаг {i}, source {j}: type должен быть text/image/table')
            if not _clean(src.get('source')):
                errs.append(f'Шаг {i}, source {j}: отсутствует source')
            if not _clean(src.get('snippet_or_summary')):
                errs.append(f'Шаг {i}, source {j}: отсутствует snippet_or_summary')
        ctx = step.get('discovery_context') or {}
        if ctx:
            geography = ctx.get('geography') or {}
            country = geography.get('country')
            city = geography.get('city')
            errs.extend(validate_qid_dict(country, f'Шаг {i}: страна'))
            errs.extend(validate_qid_dict(city, f'Шаг {i}: город'))
            if city and not country:
                errs.append(f'Шаг {i}: сначала укажите страну, затем город')
            branches = ctx.get('science_branches') or []
            for b_idx, branch in enumerate(branches, start=1):
                errs.extend(validate_qid_dict(branch, f'Шаг {i}: отрасль {b_idx}'))
    for k, edge in enumerate(doc.get('edges') or [], start=1):
        frm = int(edge.get('from_step_id') or 0)
        to = int(edge.get('to_step_id') or 0)
        if not (1 <= frm <= len(steps)) or not (1 <= to <= len(steps)):
            errs.append(f'Ребро {k}: неверные номера шагов')
        if frm == to:
            errs.append(f'Ребро {k}: петля на шаг {frm} не допускается')
        if _clean(edge.get('directionality')) not in {'directed', 'bidirectional', 'simultaneous'}:
            errs.append(f'Ребро {k}: неверная направленность')
    return errs


validate_btn = W.Button(description='Проверить YAML', icon='check')
export_btn = W.Button(description='Сохранить YAML', icon='save', button_style='success')
preview_btn = W.Button(description='Показать YAML', icon='code')


def handle_validate(_):
    doc = build_document()
    errs = validate_doc(doc)
    if errs:
        status.value = '<div class="t1-note"><b>Проверка не пройдена.</b><br>' + '<br>'.join(errs) + '</div>'
    else:
        status.value = '<div class="t1-note"><b>Проверка пройдена.</b> YAML совместим с artifact_version: 3.</div>'


def handle_preview(_):
    with export_out:
        clear_output()
        doc = build_document()
        print(yaml.safe_dump(doc, allow_unicode=True, sort_keys=False))


def handle_export(_):
    doc = build_document()
    errs = validate_doc(doc)
    if errs:
        status.value = '<div class="t1-note"><b>Сначала исправьте ошибки.</b><br>' + '<br>'.join(errs) + '</div>'
        return
    file_name = _clean(output_name.value) or f"{doc['submission_id']}.yaml"
    if not file_name.endswith('.yaml') and not file_name.endswith('.yml'):
        file_name += '.yaml'
    path = Path(file_name)
    path.write_text(yaml.safe_dump(doc, allow_unicode=True, sort_keys=False), encoding='utf-8')
    status.value = f'<div class="t1-note"><b>Файл сохранён:</b> <code>{path}</code></div>'
    with export_out:
        clear_output()
        print(path.resolve())
    if IN_COLAB and colab_files is not None:
        try:
            colab_files.download(str(path))
        except Exception:
            pass


validate_btn.on_click(handle_validate)
preview_btn.on_click(handle_preview)
export_btn.on_click(handle_export)

root = W.VBox([
    W.HTML('<div class="t1-note">Заполняйте сверху вниз. Географию открытия и отрасли можно выбрать вручную через Wikidata — именно эти QID потом попадут в YAML и дальше в автоконвейер.</div>'),
    expert_block,
    W.HTML('<div class="t1-card"><h3>Тема и домен</h3></div>'),
    topic,
    cutoff_year,
    W.HBox([domain_query, domain_search_btn, domain_select_btn]),
    domain_results,
    domain_selected,
    W.HTML('<div class="t1-card"><h3>Публикации</h3></div>'),
    W.HBox([add_paper_btn, remove_paper_btn]),
    papers_box,
    W.HTML('<div class="t1-card"><h3>Шаги траектории</h3></div>'),
    W.HBox([add_step_btn, remove_step_btn]),
    steps_box,
    W.HTML('<div class="t1-card"><h3>Связи между шагами</h3><div class="t1-small">Здесь появились и поле направленности, и текстовая метка направления ребра.</div></div>'),
    W.HBox([add_edge_btn, remove_edge_btn]),
    edges_box,
    W.HTML('<div class="t1-card"><h3>Экспорт</h3></div>'),
    output_name,
    W.HBox([validate_btn, preview_btn, export_btn]),
    status,
    export_out,
])

display(root)